In [1]:
import os
import mlflow


In [2]:
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy

from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from catboost import CatBoostRegressor


from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error

In [3]:
#загружаем очищенный датасет
df = pd.read_pickle('/home/mainuser/savvin_iis/data/clean_data.pkl')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 301 entries, 0 to 300
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype   
---  ------                --------------  -----   
 0   Car_Name              301 non-null    category
 1   Year                  301 non-null    int16   
 2   Selling_Price         301 non-null    float16 
 3   Target_Present_Price  301 non-null    float16 
 4   Driven_kms            301 non-null    int32   
 5   Fuel_Type             301 non-null    category
 6   Selling_type          301 non-null    category
 7   Transmission          301 non-null    category
dtypes: category(4), float16(2), int16(1), int32(1)
memory usage: 5.1 KB


In [5]:
df

/home/mainuser/savvin_iis/.savvin_iis/lib/python3.10/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,Car_Name,Year,Selling_Price,Target_Present_Price,Driven_kms,Fuel_Type,Selling_type,Transmission
0,ritz,2014,3.349609,5.589844,27000,Petrol,Dealer,Manual
1,sx4,2013,4.750000,9.539062,43000,Diesel,Dealer,Manual
2,ciaz,2017,7.250000,9.851562,6900,Petrol,Dealer,Manual
3,wagon r,2011,2.849609,4.148438,5200,Petrol,Dealer,Manual
4,swift,2014,4.601562,6.871094,42450,Diesel,Dealer,Manual
...,...,...,...,...,...,...,...,...
296,city,2016,9.500000,11.601562,33988,Diesel,Dealer,Manual
297,brio,2015,4.000000,5.898438,60000,Petrol,Dealer,Manual
298,city,2009,3.349609,11.000000,87934,Petrol,Dealer,Manual
299,city,2017,11.500000,12.500000,9000,Diesel,Dealer,Manual


In [6]:
#разбиваем на обучающую и тестовую
X_train, X_test, y_train, y_test = train_test_split(df.drop('Target_Present_Price', axis=1), df['Target_Present_Price'], test_size=0.25, random_state=2)

In [7]:
#выделение категориальных признаков
cat_features = X_train.select_dtypes(include=['category','object']).columns.to_list()
cat_features

['Car_Name', 'Fuel_Type', 'Selling_type', 'Transmission']

In [8]:
#Выделение числовых признаков
num_features = X_train.select_dtypes(include=['number']).columns.to_list()
num_features

['Year', 'Selling_Price', 'Driven_kms']

In [9]:
#задание скалирования, кодирования и регрессора
s_scaler = StandardScaler()
l_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=99999999) # unknown_value нужно выбирать с умом
regressor = RandomForestRegressor()

In [10]:
# Для удобной работы со столбцами
preprocessor = ColumnTransformer(
    transformers=[
        ('num', s_scaler, num_features),  # преобразования для числовых признаков
        ('cat', l_encoder, cat_features), # преобразования для категориальных признаков
    ],
    remainder='drop' ) # Удаляем столбцы, которые не затронуты преобразования

In [11]:
#обучение
pipeline = Pipeline(steps=[('preprocessor', preprocessor), 
                           ('model', regressor)])

pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [12]:
#расчёт метрик
predictions = pipeline.predict(X_test) 

metrics = {}
metrics["mae"] = mean_absolute_error(y_test, predictions)   
metrics["mape"] = mean_absolute_percentage_error(y_test, predictions)
metrics["mse"] = mean_squared_error(y_test, predictions)

metrics

{'mae': 1.2405681730571545,
 'mape': 0.8213760549427813,
 'mse': 2.7044024992355387}

In [13]:

# Работаем с MLflow локально
TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

registry_uri = f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}"
tracking_uri = f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}"

mlflow.set_tracking_uri(tracking_uri)   
mlflow.set_registry_uri(registry_uri) 

In [14]:
# название тестового эксперимента, запуска (run) внутри него, имени, под которым модель будет регистрироваться
EXPERIMENT_NAME = "estate_project"
RUN_NAME = "baseline model"
REGISTRY_MODEL_NAME = "estate_model_rf"

In [15]:
# Обязательно логируем сигнатуру модели и пример входных данных. Подготовим их
from mlflow.models import infer_signature

signature =  infer_signature(model_input = X_train.head(5))
input_example = X_train.head(5)

/home/mainuser/savvin_iis/.savvin_iis/lib/python3.10/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


In [19]:
# Будем логировать requirements и артефакт - текстовый файл
req_file = '/home/mainuser/savvin_iis/requirements.txt'
art = '/home/mainuser/savvin_iis/comments.txt'

In [20]:
# Параметры, котороые будут залогированы, можем задавать вручную или полностью взять из модели
#params_dict = {'n_estimators': 10, 'max_depth': 10}
params_dict = pipeline.get_params()

In [21]:
# Когда создаем новый эксперимент, то: 
#experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

# Впоследствии. чтобы добавлять запуски в этот же эксепримент мы должны получить его id:
experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    # получаем уникальный идентификатор запуска эксперимента
    run_id = run.info.run_id 
    mlflow.sklearn.log_model(pipeline, 
                             artifact_path="models",
                             signature=signature,
                             input_example=input_example,
                             pip_requirements=req_file
                             )
    mlflow.log_metrics(metrics)
    mlflow.log_artifact(art)
    mlflow.log_params(params_dict)

run = mlflow.get_run(run_id) 
assert (run.info.status =='FINISHED')

2026/09/29 10:34:26 INFO mlflow.tracking._tracking_service.client: 🏃 View run baseline model at: http://127.0.0.1:5000/#/experiments/1/runs/b1200a5d0569428fa446896bc1d13fa7.
2026/09/29 10:34:26 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


In [22]:
from mlflow.models import infer_signature
input_example = X_train.head(5)
signature =  infer_signature(model_input = X_train.head(5))

/home/mainuser/savvin_iis/.savvin_iis/lib/python3.10/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


In [23]:
#другая модель
regressor2 = RandomForestRegressor(n_estimators=10, max_depth=6)

In [24]:
pipeline = Pipeline(steps=[('preprocessor', preprocessor), 
                           ('model', regressor2)])

pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [25]:
predictions = pipeline.predict(X_test) 
metrics = {}
metrics["mae"] = mean_absolute_error(y_test, predictions)   
metrics["mape"] = mean_absolute_percentage_error(y_test, predictions)
metrics["mse"] = mean_squared_error(y_test, predictions)

metrics

{'mae': 1.7360775155125017,
 'mape': 1.458538550448228,
 'mse': 5.890649382506515}

In [26]:
# !!! Проверить название прогона а также все логируемые параметры и артефакты, что они соответствуют второй "маленькой" модели. 
#Логируем маленькую модель
RUN_NAME = 'smaller_model'

experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    # получаем уникальный идентификатор запуска эксперимента
    run_id = run.info.run_id 
    mlflow.sklearn.log_model(pipeline, 
                             artifact_path="models",
                             signature=signature,
                             input_example=input_example,
                             pip_requirements=req_file
                             )
    mlflow.log_metrics(metrics)
    mlflow.log_artifact(art)
    mlflow.log_params(pipeline.get_params())

run = mlflow.get_run(run_id) 
assert (run.info.status =='FINISHED')

2026/09/29 10:35:51 INFO mlflow.tracking._tracking_service.client: 🏃 View run smaller_model at: http://127.0.0.1:5000/#/experiments/1/runs/df3f9df3374d4b9e919e80ca78e1a4f7.
2026/09/29 10:35:51 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


**Feature engineerig**

In [27]:
from sklearn.preprocessing import QuantileTransformer, SplineTransformer, PolynomialFeatures, MinMaxScaler

In [28]:
X_train_sklearn = X_train.copy()

In [29]:
pf = PolynomialFeatures(degree=2)

In [30]:
X_train_sklearn

/home/mainuser/savvin_iis/.savvin_iis/lib/python3.10/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,Car_Name,Year,Selling_Price,Driven_kms,Fuel_Type,Selling_type,Transmission
155,Honda Activa 4G,2017,0.479980,4300,Petrol,Individual,Automatic
104,Royal Enfield Classic 350,2017,1.349609,4100,Petrol,Individual,Manual
285,jazz,2016,7.398438,15059,Petrol,Dealer,Automatic
219,verna,2012,4.500000,36000,Petrol,Dealer,Manual
291,brio,2015,5.398438,31427,Petrol,Dealer,Manual
...,...,...,...,...,...,...,...
75,etios g,2015,3.949219,36000,Petrol,Dealer,Manual
22,sx4,2011,4.398438,50000,Petrol,Dealer,Automatic
72,corolla altis,2013,7.449219,56001,Petrol,Dealer,Manual
15,ertiga,2016,7.750000,43000,Diesel,Dealer,Manual


In [31]:
#включение автологирования
mlflow.sklearn.autolog()

with mlflow.start_run(run_name='auto', experiment_id=experiment_id) as run:
    pipeline.fit(X_train, y_train)

2026/09/29 10:36:14 WARNING mlflow.utils.autologging_utils: MLflow sklearn autologging is known to be compatible with 0.24.1 <= scikit-learn <= 1.5.1, but the installed version is 1.7.2. If you encounter errors during autologging, try upgrading / downgrading scikit-learn to a compatible version, or try upgrading MLflow.
2026/09/29 10:36:15 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/home/mainuser/savvin_iis/.savvin_iis/lib/python3.10/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever th

In [32]:
#отключение автологирования - не понравилось
mlflow.sklearn.autolog(disable=True)

In [36]:
pf.fit_transform(X_train_sklearn[['Selling_Price','Driven_kms']])

array([[1.00000000e+00, 4.79980469e-01, 4.30000000e+03, 2.30381250e-01,
        2.06391602e+03, 1.84900000e+07],
       [1.00000000e+00, 1.34960938e+00, 4.10000000e+03, 1.82144547e+00,
        5.53339844e+03, 1.68100000e+07],
       [1.00000000e+00, 7.39843750e+00, 1.50590000e+04, 5.47368774e+01,
        1.11413070e+05, 2.26773481e+08],
       ...,
       [1.00000000e+00, 7.44921875e+00, 5.60010000e+04, 5.54908600e+01,
        4.17163699e+05, 3.13611200e+09],
       [1.00000000e+00, 7.75000000e+00, 4.30000000e+04, 6.00625000e+01,
        3.33250000e+05, 1.84900000e+09],
       [1.00000000e+00, 4.19921875e-01, 1.20000000e+04, 1.76334381e-01,
        5.03906250e+03, 1.44000000e+08]], shape=(225, 6))

In [37]:
sp = SplineTransformer(n_knots=3, degree=3)
sp.fit_transform(X_train_sklearn[['Driven_kms']])

array([[1.59174223e-01, 6.66436925e-01, 1.74388264e-01, 5.87060755e-07,
        0.00000000e+00],
       [1.59562848e-01, 6.66460389e-01, 1.73976264e-01, 4.99159983e-07,
        0.00000000e+00],
       [1.39185616e-01, 6.63367490e-01, 1.97413878e-01, 3.30161856e-05,
        0.00000000e+00],
       ...,
       [7.84167130e-02, 6.22769436e-01, 2.96984763e-01, 1.82908835e-03,
        0.00000000e+00],
       [9.52392302e-02, 6.40172664e-01, 2.63766811e-01, 8.21294755e-04,
        0.00000000e+00],
       [1.44687491e-01, 6.64595243e-01, 1.90700995e-01, 1.62714322e-05,
        0.00000000e+00]], shape=(225, 5))

In [39]:
qt = QuantileTransformer()
qt.fit_transform(X_train_sklearn[['Driven_kms']])

/home/mainuser/savvin_iis/.savvin_iis/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:2846: UserWarning: n_quantiles (1000) is greater than the total number of samples (225). n_quantiles is set to n_samples.
  warnings.warn(


array([[0.06696429],
       [0.0625    ],
       [0.25892857],
       [0.55803571],
       [0.46205357],
       [0.99107143],
       [0.83482143],
       [0.21428571],
       [0.52232143],
       [0.71428571],
       [0.39732143],
       [0.70089286],
       [0.57142857],
       [0.41294643],
       [0.71428571],
       [0.63392857],
       [0.57589286],
       [0.17857143],
       [0.11383929],
       [0.9375    ],
       [0.42857143],
       [0.87946429],
       [0.53571429],
       [0.98660714],
       [0.15625   ],
       [0.1875    ],
       [0.3125    ],
       [0.77008929],
       [0.00892857],
       [0.54910714],
       [0.79464286],
       [0.36607143],
       [0.47544643],
       [0.20089286],
       [0.09375   ],
       [0.125     ],
       [0.77008929],
       [0.29910714],
       [0.45535714],
       [0.23883929],
       [0.61383929],
       [0.01785714],
       [0.44866071],
       [0.10267857],
       [0.02678571],
       [0.04464286],
       [0.13392857],
       [0.968

In [40]:
pf = PolynomialFeatures(degree=2)
qt = QuantileTransformer()
sp = SplineTransformer(n_knots=3, degree=3)

In [41]:
# Значения преобразованных признаков нужно отскейлить, поэтому создаем pipeline из двух шагов - преобразование и скейлинг
pf_pipeline = Pipeline(steps=[
    ('poly', pf),
    ('scale', StandardScaler())
])

In [46]:
preprocessor_sklearn = ColumnTransformer(
    transformers=[
        ('num', s_scaler, num_features),  # преобразования для числовых признаков
        ('cat', l_encoder, cat_features), # преобразования для категориальных признаков
        ('quantile', qt,num_features),
        ('poly', pf_pipeline, ['Selling_Price','Driven_kms']), # В преобразования добавляем созданный ранее pipeline
        ('spline', sp, ['Driven_kms'])
    ],
    remainder='drop',
    ) # Удаляем столбцы, которые не затронуты преобразования

In [47]:
## не влезаем в float64 в полиномальном преобразовании. Использовать его нужно с умом!
X_train_sklearn[['Selling_Price','Driven_kms']] = X_train_sklearn[['Selling_Price','Driven_kms']].astype('float64')
X_train_sklearn[['Selling_Price','Driven_kms']] = X_train_sklearn[['Selling_Price','Driven_kms']].astype('float64')

In [48]:
X_train_sklearn_raw = preprocessor_sklearn.fit_transform(X_train_sklearn)
X_train_sklearn = pd.DataFrame(X_train_sklearn_raw, columns=preprocessor_sklearn.get_feature_names_out())

/home/mainuser/savvin_iis/.savvin_iis/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:2846: UserWarning: n_quantiles (1000) is greater than the total number of samples (225). n_quantiles is set to n_samples.
  warnings.warn(


In [49]:
# Удобно использовать для отображения всех строк\столбцов в DataFrame
with pd.option_context('display.max_rows', 5, 'display.max_columns', None):
    display (X_train_sklearn)

,num__Year,num__Selling_Price,num__Driven_kms,cat__Car_Name,cat__Fuel_Type,cat__Selling_type,cat__Transmission,quantile__Year,quantile__Selling_Price,quantile__Driven_kms,poly__1,poly__Selling_Price,poly__Driven_kms,poly__Selling_Price^2,poly__Selling_Price Driven_kms,poly__Driven_kms^2,spline__Driven_kms_sp_0,spline__Driven_kms_sp_1,spline__Driven_kms_sp_2,spline__Driven_kms_sp_3,spline__Driven_kms_sp_4
0,1.171606,-0.819553,-0.797744,22.0,2.0,1.0,0.0,0.941964,0.127232,0.066964,0.0,-0.819553,-0.797744,-0.376286,-0.636605,-0.186065,0.159174,0.666437,0.174388,5.870608e-07,0.0
1,1.171606,-0.657343,-0.802502,34.0,2.0,1.0,1.0,0.941964,0.301339,0.062500,0.0,-0.657343,-0.802502,-0.364830,-0.624404,-0.186163,0.159563,0.666460,0.173976,4.991600e-07,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
223,0.828808,0.536510,0.122902,58.0,1.0,0.0,1.0,0.792411,0.845982,0.680804,0.0,0.536510,0.122902,0.054486,0.528061,-0.078957,0.095239,0.640173,0.263767,8.212948e-04,0.0
224,-0.199584,-0.830756,-0.614567,25.0,2.0,1.0,1.0,0.352679,0.104911,0.200893,0.0,-0.830756,-0.614567,-0.376675,-0.626143,-0.178721,0.144687,0.664595,0.190701,1.627143e-05,0.0


In [50]:
pipeline_sklearn = Pipeline(steps=[
    ('transform', preprocessor_sklearn),
    ('model', regressor)
])

model_sklearn = pipeline_sklearn.fit(X_train, y_train)

/home/mainuser/savvin_iis/.savvin_iis/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:2846: UserWarning: n_quantiles (1000) is greater than the total number of samples (225). n_quantiles is set to n_samples.
  warnings.warn(


In [51]:
model_sklearn

,steps,"[('transform', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [52]:
predictions = model_sklearn.predict(X_test) 
metrics = {}
metrics["mae"] = mean_absolute_error(y_test, predictions)   
metrics["mape"] = mean_absolute_percentage_error(y_test, predictions)
metrics["mse"] = mean_squared_error(y_test, predictions)

metrics

{'mae': 1.277165784333882,
 'mape': 0.7652950486366265,
 'mse': 2.883648229837261}

In [53]:
experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
RUN_NAME = 'fe_sklearn1'

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    # получаем уникальный идентификатор запуска эксперимента
    run_id = run.info.run_id 
    mlflow.sklearn.log_model(model_sklearn, 
                             artifact_path="models",
                             signature=signature,
                             input_example=input_example,
                             pip_requirements=req_file
                             )
    mlflow.log_metrics(metrics)
    mlflow.log_artifact(art)
    mlflow.log_params(model_sklearn.get_params())

run = mlflow.get_run(run_id) 
assert (run.info.status =='FINISHED')

2026/09/29 10:42:02 INFO mlflow.tracking._tracking_service.client: 🏃 View run fe_sklearn1 at: http://127.0.0.1:5000/#/experiments/1/runs/6a9d91c1902b4e309783d6bd1b6105c5.
2026/09/29 10:42:02 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


**Используем sklearn признаки для feature  selection**


In [76]:
from sklearn.feature_selection import RFE

In [77]:
rfe_skl_selector = RFE(estimator=regressor, n_features_to_select=12, step = 0.2) #drop 20% of features each iteration
X_train_skl_rfe = rfe_skl_selector.fit_transform(X_train_sklearn,y_train)

In [78]:
X_train_skl_rfe = pd.DataFrame(X_train_skl_rfe, columns=rfe_skl_selector.get_feature_names_out())
X_train_skl_rfe

,num__Year,num__Selling_Price,num__Driven_kms,cat__Car_Name,cat__Transmission,quantile__Year,quantile__Selling_Price,quantile__Driven_kms,poly__Selling_Price,poly__Selling_Price^2,poly__Selling_Price Driven_kms,poly__Driven_kms^2
0,1.171606,-0.819553,-0.797744,22.0,0.0,0.941964,0.127232,0.066964,-0.819553,-0.376286,-0.636605,-0.186065
1,1.171606,-0.657343,-0.802502,34.0,1.0,0.941964,0.301339,0.062500,-0.657343,-0.364830,-0.624404,-0.186163
2,0.828808,0.470933,-0.541795,69.0,0.0,0.792411,0.816964,0.258929,0.470933,0.016143,-0.252062,-0.173878
3,-0.542382,-0.069707,-0.043623,75.0,1.0,0.247768,0.573661,0.558036,-0.069707,-0.232151,-0.074166,-0.111314
4,0.486011,0.097877,-0.152412,49.0,1.0,0.604911,0.689732,0.462054,0.097877,-0.168123,-0.047240,-0.129356
...,...,...,...,...,...,...,...,...,...,...,...,...
220,0.486011,-0.172443,-0.043623,60.0,1.0,0.604911,0.511161,0.558036,-0.172443,-0.265656,-0.143894,-0.111314
221,-0.885179,-0.088651,0.289428,74.0,0.0,0.174107,0.555804,0.770089,-0.088651,-0.238658,0.129525,-0.040865
222,-0.199584,0.480405,0.432187,53.0,1.0,0.352679,0.823661,0.825893,0.480405,0.021572,0.823156,-0.003644
223,0.828808,0.536510,0.122902,58.0,1.0,0.792411,0.845982,0.680804,0.536510,0.054486,0.528061,-0.078957


In [79]:
rfe_cols = X_train_skl_rfe.columns.tolist()
rfe_cols


['num__Year',
 'num__Selling_Price',
 'num__Driven_kms',
 'cat__Car_Name',
 'cat__Transmission',
 'quantile__Year',
 'quantile__Selling_Price',
 'quantile__Driven_kms',
 'poly__Selling_Price',
 'poly__Selling_Price^2',
 'poly__Selling_Price Driven_kms',
 'poly__Driven_kms^2']

In [80]:
rfe_idx = rfe_skl_selector.support_
rfe_idx

array([ True,  True,  True,  True, False, False,  True,  True,  True,
        True, False,  True, False,  True,  True,  True, False, False,
       False, False, False])

In [81]:
# Отбираемые столбцы нужно залогировать, иначе мы потеряем информацию о том, какие призныки выбраны
with open('rfe_skl_idx.txt', 'w+') as f:
    f.write(str(rfe_idx))
with open('rfe_skl_cols.txt', 'w+') as f:
    f.write(str(rfe_cols))

In [82]:
class ColumnExtractor(object):

    def __init__(self, cols):
        self.cols = cols

    def transform(self, X):
        return X[:,self.cols]
    
    def fit(self, X, y=None):
        return self

In [83]:
rfe_skl_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_sklearn), 
    ('rfe_extractor', ColumnExtractor(rfe_idx)),
    ('model', regressor)
])

rfe_skl_pipeline.fit(X_train, y_train)

/home/mainuser/savvin_iis/.savvin_iis/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:2846: UserWarning: n_quantiles (1000) is greater than the total number of samples (225). n_quantiles is set to n_samples.
  warnings.warn(


,steps,"[('preprocessor', ...), ('rfe_extractor', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [84]:
#логируем модель с отобранными признаками склерна
predictions_rfe_skl = rfe_skl_pipeline.predict(X_test)

metrics = {}
metrics["mae"] = mean_absolute_error(y_test, predictions_rfe_skl)   
metrics["mape"] = mean_absolute_percentage_error(y_test, predictions_rfe_skl)
metrics["mse"] = mean_squared_error(y_test, predictions_rfe_skl)

metrics
experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
RUN_NAME = 'rfe_skl_feature_selection'

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    # получаем уникальный идентификатор запуска эксперимента
    run_id = run.info.run_id 
    mlflow.sklearn.log_model(rfe_skl_pipeline, 
                             artifact_path="models",
                             signature=signature,
                             input_example=input_example,
                             pip_requirements=req_file
                             )
    mlflow.log_metrics(metrics)
    mlflow.log_artifact('rfe_skl_cols.txt')
    mlflow.log_artifact('rfe_skl_idx.txt')
    mlflow.log_params(model_sklearn.get_params())

run = mlflow.get_run(run_id) 
assert (run.info.status =='FINISHED')

2026/09/29 10:49:05 INFO mlflow.tracking._tracking_service.client: 🏃 View run rfe_skl_feature_selection at: http://127.0.0.1:5000/#/experiments/1/runs/53f4fd18646047e3b48fb2458d8df4c9.
2026/09/29 10:49:05 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


**HYPERPARAMS**


In [72]:
##Gridsearch
from sklearn.model_selection import  GridSearchCV

In [73]:
#подбираем параметры
param_grid = {
    'model__max_depth': [i for i in range(1,10)],
    'model__n_estimators':[i for i in range(1,5)],
    'model__max_features':[i/10 for i in range(1,10)]
}

In [90]:
gs = GridSearchCV(rfe_skl_pipeline, param_grid, cv=2, scoring='neg_mean_absolute_error')
gs.fit(X_train, y_train)
print("Лучшие гиперпараметры:", gs.best_params_)

/home/mainuser/savvin_iis/.savvin_iis/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:2846: UserWarning: n_quantiles (1000) is greater than the total number of samples (112). n_quantiles is set to n_samples.
  warnings.warn(
/home/mainuser/savvin_iis/.savvin_iis/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:2846: UserWarning: n_quantiles (1000) is greater than the total number of samples (113). n_quantiles is set to n_samples.
  warnings.warn(
/home/mainuser/savvin_iis/.savvin_iis/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:2846: UserWarning: n_quantiles (1000) is greater than the total number of samples (112). n_quantiles is set to n_samples.
  warnings.warn(
/home/mainuser/savvin_iis/.savvin_iis/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:2846: UserWarning: n_quantiles (1000) is greater than the total number of samples (113). n_quantiles is set to n_samples.
  warnings.warn(
/home/mainuser/savvin_iis/.savvin_iis/lib/python3.10

Лучшие гиперпараметры: {'model__max_depth': 5, 'model__max_features': 0.5, 'model__n_estimators': 4}


/home/mainuser/savvin_iis/.savvin_iis/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:2846: UserWarning: n_quantiles (1000) is greater than the total number of samples (112). n_quantiles is set to n_samples.
  warnings.warn(
/home/mainuser/savvin_iis/.savvin_iis/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:2846: UserWarning: n_quantiles (1000) is greater than the total number of samples (113). n_quantiles is set to n_samples.
  warnings.warn(
/home/mainuser/savvin_iis/.savvin_iis/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:2846: UserWarning: n_quantiles (1000) is greater than the total number of samples (112). n_quantiles is set to n_samples.
  warnings.warn(
/home/mainuser/savvin_iis/.savvin_iis/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:2846: UserWarning: n_quantiles (1000) is greater than the total number of samples (113). n_quantiles is set to n_samples.
  warnings.warn(
/home/mainuser/savvin_iis/.savvin_iis/lib/python3.10

In [92]:
#Обучаем и логируем модель с лучшими параметрами
regressor1=RandomForestRegressor(max_depth=5, max_features=0.5, n_estimators=4)
gs_pipeline  = Pipeline(steps=[
    ('preprocessor', preprocessor_sklearn), 
    ('rfe_extractor', ColumnExtractor(rfe_idx)),
    ('model', regressor1)
])
gs_pipeline.fit(X_train,y_train)
predictions_gs = gs_pipeline.predict(X_test)
metrics = {}
metrics["mae"] = mean_absolute_error(y_test, predictions_gs)   
metrics["mape"] = mean_absolute_percentage_error(y_test, predictions_gs)
metrics["mse"] = mean_squared_error(y_test, predictions_gs)

metrics
# Проведем стандартную проверку на тестовом множестве и залогируем run
experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
RUN_NAME = 'gridsearch'

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    # получаем уникальный идентификатор запуска эксперимента
    run_id = run.info.run_id 
    mlflow.sklearn.log_model(gs_pipeline, 
                             artifact_path="models",
                             signature=signature,
                             input_example=input_example,
                             pip_requirements=req_file
                             )
    mlflow.log_metrics(metrics)
    mlflow.log_artifact(art)
    mlflow.log_params(gs_pipeline.get_params())

run = mlflow.get_run(run_id) 
assert (run.info.status =='FINISHED')

/home/mainuser/savvin_iis/.savvin_iis/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:2846: UserWarning: n_quantiles (1000) is greater than the total number of samples (225). n_quantiles is set to n_samples.
  warnings.warn(


2026/09/29 11:01:58 INFO mlflow.tracking._tracking_service.client: 🏃 View run gridsearch at: http://127.0.0.1:5000/#/experiments/1/runs/7da9db14097f42199e0efc53ec4fe516.
2026/09/29 11:01:58 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


In [97]:
#выделение всей выборки
Y = df['Target_Present_Price']
X = df.drop('Target_Present_Price', axis=1)

In [104]:
#использование лучшей модeли на всей выборке
gs_pipeline.fit(X,Y)
predictions_gs = gs_pipeline.predict(X_test)
metrics = {}
metrics["mae"] = mean_absolute_error(y_test, predictions_gs)   
metrics["mape"] = mean_absolute_percentage_error(y_test, predictions_gs)
metrics["mse"] = mean_squared_error(y_test, predictions_gs)

metrics
# Проведем стандартную проверку на тестовом множестве и залогируем run
experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
RUN_NAME = 'production'

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    # получаем уникальный идентификатор запуска эксперимента
    run_id = run.info.run_id 
    mlflow.sklearn.log_model(gs_pipeline, 
                             artifact_path="models",
                             signature=signature,
                             input_example=input_example,
                             pip_requirements=req_file
                             )
    mlflow.log_metrics(metrics)
    mlflow.log_artifact(art)
    mlflow.log_params(gs_pipeline.get_params())

run = mlflow.get_run(run_id) 
assert (run.info.status =='FINISHED')


/home/mainuser/savvin_iis/.savvin_iis/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:2846: UserWarning: n_quantiles (1000) is greater than the total number of samples (301). n_quantiles is set to n_samples.
  warnings.warn(


2026/09/29 11:05:29 INFO mlflow.tracking._tracking_service.client: 🏃 View run production at: http://127.0.0.1:5000/#/experiments/1/runs/bbe77da6ca234e1fa503dbbdd17299d7.
2026/09/29 11:05:29 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.
